# axon-lang — treinar os experts (Colab)

Constrói os experts do pyaxon: `ModularRouter` (escolhe a família da pergunta) +
`SparseKB` (recupera a lição). É a metade de **busca** do sistema. A metade que
**escreve** a resposta é o `finetune_expert_colab.ipynb`.

O notebook é linear — rode de cima pra baixo:

| | seção | quando usar |
|---|---|---|
| 1 | Preparar o ambiente | uma vez por sessão |
| 2 | Treinar **um** expert | trocando o nome a cada rodada |
| 3 | Conferir o expert | logo depois de treinar |
| 4 | Treinar **todos** de uma vez | no lugar de repetir 2 e 3 |
| 5 | Validar o sistema inteiro | com todos os experts prontos |
| 6 | Ajustar o `alpha` | opcional |
| 7 | Perguntar ao sistema | quando quiser usar |

> **Antes de começar:** `Ambiente de execução → Alterar o tipo de ambiente → T4 GPU`.

## 1. Preparar o ambiente

Clona o repositório (ou atualiza, se já estiver lá), compila o `_axon.so` e confere que
o pyaxon importa. Uma vez por sessão — enquanto ela viver, não precisa repetir.

`CUDA = True` liga o backend de GPU. Vale a pena: o `LinearRouter` de cada família é uma
rede de verdade (`_axon.nn`), então o forward e o backward do treino passam pelo
`ops::matmul`, que despacha pra GPU sozinho acima do limiar padrão. Não precisa chamar
nada — basta compilar assim. O custo é a compilação demorar mais.

> Se você recompilar depois de já ter importado o pyaxon nesta sessão, **reinicie o
> ambiente** antes: o Python não recarrega extensão C que já está na memória.

In [ ]:
CUDA = True   # False = só CPU (compila rápido, treina devagar)

%cd /content
import os, subprocess, sys

REPO = "https://github.com/geraldogrise/axon-llm.git"
if os.path.isdir("axon-llm"):
    !git -C axon-llm pull --quiet && echo "repo atualizado"
else:
    !git clone --depth 1 $REPO axon-llm

!apt-get -qq install -y ninja-build > /dev/null
!pip -q install pybind11 numpy

import pybind11
cfg = ["cmake", "-S", ".", "-B", "build-colab", "-G", "Ninja",
       "-DCMAKE_BUILD_TYPE=Release",
       "-DAXON_BUILD_PYTHON=ON", "-DAXON_BUILD_TESTS=OFF", "-DAXON_BUILD_EXAMPLES=OFF",
       "-DAXON_ENABLE_NATIVE=OFF",
       f"-DAXON_ENABLE_CUDA={'ON' if CUDA else 'OFF'}",
       f"-DAXON_USE_CUBLAS={'ON' if CUDA else 'OFF'}",
       f"-Dpybind11_DIR={pybind11.get_cmake_dir()}"]
for c in (cfg, ["cmake", "--build", "build-colab", "-j"]):
    p = subprocess.run(c, cwd="axon-llm", capture_output=True, text=True)
    print(p.stdout[-600:] or p.stderr[-600:])
    assert p.returncode == 0, "build falhou -- veja o log acima"

sys.path.insert(0, "/content/axon-llm/notebooks")
sys.path.insert(0, "/content/axon-llm/python")
import pyaxon as ax
import axon_colab as ac

print("\npyaxon ok | cuda_available:", ax.cuda_available(),
      "|", ax.cuda_device_name() if ax.cuda_available() else "sem GPU")

## 2. Treinar um expert

Troque o `EXPERT` e rode. A célula baixa **só** a branch de dados daquele expert,
treina e salva em `MyDrive/axon_experts/`. No fim ela imprime a acurácia de
roteamento nas perguntas de teste do próprio expert.

```
escolar   java     dotnet   js       python   php      rust     go       ruby
aws       azure    gcp      oci      docker   git      kubernetes        web
terminal  (bash + shell juntos -- ver seção 8)
```

Cada expert é independente: treinar `rust` não mexe em nada do `go`.

In [ ]:
EXPERT = "go"

ac.tabela()   # o mapa completo: expert -> branch -> script

saida = ac.build_expert(EXPERT, repo="axon-llm", extra_env={"AXON_EPOCHS": "300"})
destino = ac.salvar_expert(saida, EXPERT)

## 3. Conferir o expert

Recarrega o que foi salvo e roda as perguntas de teste **daquele** expert — cada
`build_*_experts.py` traz a própria lista, com a família correta esperada de cada
pergunta. Trocar o `EXPERT` troca as perguntas junto.

In [ ]:
router = ax.modular.ModularRouter().load(os.path.join(saida, "router"))
kb = ax.vindex.SparseKB().load(os.path.join(saida, "kb.sparse.json.gz"))

testes = ac.perguntas(EXPERT, repo="axon-llm")
ok = 0
for familia, q in testes:
    rota = router.route(q)
    acertou = rota[:1] == [familia]
    ok += acertou
    trechos = kb.retrieve(q, path_prefix=rota, top_k=1)
    snippet = trechos[0][0][:150].replace("\n", " ") if trechos else "(nada)"
    print(f"[{'OK' if acertou else 'X '}] {' > '.join(rota):26} | {q}")
    print(f"        {snippet} ...")

print(f"\nacurácia de família: {ok}/{len(testes)} = {ok / len(testes):.0%}")

## 4. Treinar todos de uma vez

No lugar de repetir as seções 2 e 3 dezenove vezes. Treina a fila **em sequência** —
um expert por vez, salvando cada um antes de começar o próximo.

- pula o que já está no Drive, então dá pra rodar de novo depois de a sessão cair;
- um expert que falhar não derruba a fila: o erro é registrado e ela continua;
- no fim imprime treinados, pulados e falhados.

In [ ]:
# ac.ORDEM_FASES   -> fase-1 até fase-12
# ac.ORDEM_TAMANHO -> do menor pro maior (rust, go, ... escolar por último)
resumo = ac.treinar_fila(ac.ORDEM_FASES, repo="axon-llm",
                         extra_env={"AXON_EPOCHS": "300"})

# um subconjunto, se preferir:
# resumo = ac.treinar_fila(["rust", "go", "ruby"], repo="axon-llm")

## 5. Validar o sistema inteiro

Aqui é medida a peça que só existe quando há vários experts: o **gate de domínio**,
que escolhe *qual* expert responde antes de rotear dentro dele.

Nas seções 2 e 3 a pergunta já chegava sabendo de qual expert veio. No uso real não —
o sistema tem que escolher entre todos. Se errar o expert, a resposta certa fica
inalcançável por melhor que aquele expert seja.

Esta célula passa **todas** as perguntas de teste pelo sistema completo e conta quantas
chegaram no expert certo. Ela testa só os experts que estão de fato no Drive.

In [ ]:
import collections

sistema = ax.system.AxonSystem.load("/content/drive/MyDrive/axon_experts")
carregados = {e.name for e in sistema.experts}
print(f"{len(carregados)} experts carregados\n")
for e in sorted(sistema.experts, key=lambda x: x.name):
    print(f"  {e.name:<22} {len(e.kb.texts):>6} passagens")

acertos, totais = collections.Counter(), collections.Counter()
confusao = collections.Counter()

for nome in sorted(ac.EXPERTS):
    alvo = ac.SAIDA[nome]
    if alvo not in carregados:          # não treinado, ou substituído pelo terminal
        continue
    for _, q in ac.perguntas(nome, repo="axon-llm"):
        escolhido, _ = sistema.route(q)
        totais[alvo] += 1
        if escolhido.name == alvo:
            acertos[alvo] += 1
        else:
            confusao[(alvo, escolhido.name)] += 1

tot, ok = sum(totais.values()), sum(acertos.values())
print(f"\ngate de domínio: {ok}/{tot} = {ok / tot:.1%}\n")

for alvo in sorted(totais, key=lambda a: acertos[a] / totais[a]):
    print(f"  {alvo:<22} {acertos[alvo]:>3}/{totais[alvo]:<3} "
          f"{acertos[alvo] / totais[alvo]:>6.0%}")

print("\nconfusões mais comuns:")
for (a, b), n in confusao.most_common(10):
    print(f"  {a:<22} -> {b:<22} {n}")

## 6. Ajustar o `alpha` (opcional)

O `route()` combina dois sinais: a recuperação (cosseno da melhor passagem) e um
classificador de palavras. O `alpha` pesa os dois — `0` é só recuperação, `1` é só
o classificador, e o padrão é `0.5`.

Não retreina nada, só re-roteia. Se algum valor bater o padrão, passe-o adiante:
`sistema.route(pergunta, alpha=0.75)`.

In [ ]:
testes = [(ac.SAIDA[n], q) for n in sorted(ac.EXPERTS)
          if ac.SAIDA[n] in carregados
          for _, q in ac.perguntas(n, repo="axon-llm")]

for alpha in (0.0, 0.25, 0.5, 0.75, 1.0):
    ok = sum(sistema.route(q, alpha=alpha)[0].name == alvo for alvo, q in testes)
    print(f"alpha={alpha:<5} {ok}/{len(testes)} = {ok / len(testes):.1%}")

## 7. Perguntar ao sistema

`route()` devolve **quem** responde; `answer()` devolve a **resposta**. O `answer()`
faz o caminho todo: escolhe o expert, recupera as passagens e monta o texto.

O campo `mode` diz como a resposta foi produzida:

- `extractive` — o trecho da lição recuperada, quase cru. É o que sai aqui, porque
  não há nenhum modelo de linguagem rodando no Colab.
- `generated` — resposta redigida. Exige um LLM; é isso que o `finetune_expert_colab.ipynb`
  vai fornecer.
- `abstain` — nada relevante o bastante, e ele diz isso em vez de inventar.

In [ ]:
for pergunta in ["como usar goroutines e canais?",
                 "o que é o determinante de uma matriz?",
                 "como escrever um dockerfile?"]:
    r = sistema.answer(pergunta)
    print(f"P: {pergunta}")
    print(f"   [{r['expert']} · {r['mode']}]")
    print(f"R: {r['answer'][:400]}")
    print()

## 8. Nota: o expert `terminal`

`bash` e `shell` são o mesmo domínio partido em dois. Separados, disputam o mesmo
vocabulário e nenhum tem termo próprio suficiente — na validação com 19 experts o
`bash` ficou em 62%, errando espalhado por `dotnet`, `go` e `python`, que é a assinatura
de um domínio sem vocabulário próprio.

O `terminal` junta os dois: 9 famílias, 54 lições, sem colisão de nome de família.

```python
saida = ac.build_expert("terminal", repo="axon-llm")
ac.salvar_expert(saida, "terminal")
```

**Depois de treinar, apague os dois antigos** — isto é obrigatório, não é limpeza:

```python
!rm -rf /content/drive/MyDrive/axon_experts/bash_experts
!rm -rf /content/drive/MyDrive/axon_experts/shell_experts
```

O `AxonSystem` adota toda pasta que tenha `router.gate.json` + `kb.sparse.json.gz`. Com
os três lá, o mesmo conteúdo fica em três experts e o gate divide o voto entre eles — a
ambiguidade que a fusão resolve volta pior. Por isso o `terminal` está fora das filas da
seção 4: ele substitui dois experts, não se soma a eles.

---

### Próximo passo

Com os experts prontos e validados, o sistema já **acha** o material certo. O que falta
é ele **escrever** a resposta: `finetune_expert_colab.ipynb`.